# Gradient descent cho phí ship Olist

Olist tính `freight_value` (phí ship của một món hàng, đơn vị BRL) theo khối lượng, kích thước và
quãng đường. Notebook này hồi quy `freight_value` theo **khối lượng (kg)** và **thể tích (lít)**
bằng gradient descent tự viết, rồi đối chiếu với normal equation (bài Basic 02).

Bạn sẽ thấy tận mắt ba điều của phần lý thuyết: learning rate có ngưỡng 2/L, chuẩn hoá feature làm
gradient descent nhanh hơn hàng trăm lần (condition number), và learning rate schedule.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/01-math-for-ml/exercises/intermediate/01-freight-gradient-descent -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Bảng dữ liệu

Mỗi dòng là một order item. Feature: `weight_kg = product_weight_g / 1000`,
`volume_l = length × height × width / 1000` (cm³ → lít). Bỏ item có sản phẩm thiếu khối lượng hoặc
kích thước.

Chia theo **thời điểm mua** (quy ước chung của track): `train` là đơn mua trước 2018-01-01, `val` từ
2018-01-01 tới trước 2018-04-01, `test` từ 2018-04-01 tới hết 2018-08-31. Bỏ đơn mua từ 2018-09-01.

In [ ]:
FEATURES = ["weight_kg", "volume_l"]
DIMS = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]


def build_freight_table(items: pd.DataFrame, products: pd.DataFrame, orders: pd.DataFrame) -> pd.DataFrame:
    """Cột: freight_value, weight_kg, volume_l, split ("train" / "val" / "test"), order_purchase_timestamp."""
    df = (items[["order_id", "product_id", "freight_value"]]
          .merge(products[["product_id", *DIMS]], on="product_id")
          .merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id")
          .dropna(subset=DIMS))
    t = pd.to_datetime(df["order_purchase_timestamp"])
    df = df[t < "2018-09-01"].copy()
    t = t[t < "2018-09-01"]
    df["weight_kg"] = df["product_weight_g"] / 1000
    df["volume_l"] = df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"] / 1000
    df["split"] = np.select([t < "2018-01-01", t < "2018-04-01"], ["train", "val"], default="test")
    return df[["freight_value", *FEATURES, "split", "order_purchase_timestamp"]].reset_index(drop=True)

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
raw = ROOT / "data" / "raw"
items = pd.read_csv(raw / "olist_order_items_dataset.csv")
products = pd.read_csv(raw / "olist_products_dataset.csv")
orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
table = build_freight_table(items, products, orders)
print(table.split.value_counts())
table[["freight_value", *FEATURES]].describe()

In [ ]:
# explore: freight_value tăng theo khối lượng nhưng rất nhiễu (quãng đường không có trong feature)
import matplotlib.pyplot as plt

train = table[table.split == "train"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, FEATURES):
    ax.scatter(train[col], train.freight_value, s=2, alpha=0.2)
    ax.set_xlabel(col)
    ax.set_ylabel("freight_value")
plt.tight_layout()
plt.show()
print(train[["freight_value", *FEATURES]].corr().round(3))

## 2. Learning rate schedule

`lr_at(t, lr0, schedule, decay, step_size)` trả learning rate cho bước cập nhật thứ `t` (đếm từ 0):

| schedule | công thức |
| --- | --- |
| `constant` | `lr0` |
| `step` | `lr0 * decay ** (t // step_size)` |
| `inverse_time` | `lr0 / (1 + decay * t)` |
| `exponential` | `lr0 * exp(-decay * t)` |

Tên khác: `raise ValueError`.

In [ ]:
def lr_at(t: int, lr0: float, schedule: str = "constant", decay: float = 0.0, step_size: int = 100) -> float:
    if schedule == "constant":
        return lr0
    if schedule == "step":
        return lr0 * decay ** (t // step_size)
    if schedule == "inverse_time":
        return lr0 / (1 + decay * t)
    if schedule == "exponential":
        return lr0 * float(np.exp(-decay * t))
    raise ValueError(f"schedule không hỗ trợ: {schedule}")

## 3. Gradient descent

Loss là MSE, $$L(w) = \frac{1}{n}\lVert Xw - y\rVert^2$$, gradient $$\nabla L = \frac{2}{n} X^\top (Xw - y)$$.
`X` đã có cột intercept. Khởi tạo `w = 0`. Trả `(w, losses)` với `losses[0]` là loss tại `w = 0` và
`losses[t]` là loss sau `t` lần cập nhật (độ dài `n_iter + 1`). Bước cập nhật thứ `t` dùng
`lr_at(t, ...)`.

In [ ]:
def gradient_descent(X: np.ndarray, y: np.ndarray, lr0: float, n_iter: int, schedule: str = "constant",
                     decay: float = 0.0, step_size: int = 100) -> tuple[np.ndarray, np.ndarray]:
    n = len(y)
    w = np.zeros(X.shape[1])
    losses = np.empty(n_iter + 1)
    r = X @ w - y
    losses[0] = r @ r / n
    for t in range(n_iter):
        w = w - lr_at(t, lr0, schedule, decay, step_size) * (2 / n) * (X.T @ r)
        r = X @ w - y
        losses[t + 1] = r @ r / n
    return w, losses

## 4. Chuẩn hoá feature và đổi hệ số về đơn vị gốc

`standardize_fit(X)` trả `(mu, sd)` theo cột (`ddof=0`). Model học trên $$z_j = (x_j - \mu_j)/\sigma_j$$
với hệ số `w_std = [b, w_1, ..., w_d]`. `to_original_scale(w_std, mu, sd)` trả hệ số tương đương trên
`x` gốc: slope $$w_j / \sigma_j$$, intercept $$b - \sum_j w_j \mu_j / \sigma_j$$.

In [ ]:
def standardize_fit(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    return X.mean(axis=0), X.std(axis=0)


def to_original_scale(w_std: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    slopes = w_std[1:] / sd
    return np.concatenate([[w_std[0] - slopes @ mu], slopes])

## 5. Thí nghiệm trên dữ liệu thật

Ba câu hỏi, trả lời bằng các cell `explore` dưới đây:

1. Trên đơn vị gốc (kg, lít) thì gradient descent cần bao nhiêu bước? Trên feature đã chuẩn hoá?
   Liên hệ với condition number của Hessian $$H = \frac{2}{n}X^\top X$$.
2. Learning rate vượt $$2/\lambda_{\max}(H)$$ thì chuyện gì xảy ra?
3. Schedule nào giúp được khi lr ban đầu hơi lớn?

In [ ]:
# explore: Hessian, ngưỡng learning rate và số bước tới hội tụ, đơn vị gốc vs chuẩn hoá
X, y = train[FEATURES].to_numpy(), train.freight_value.to_numpy()
A_raw = np.column_stack([np.ones(len(X)), X])
mu, sd = standardize_fit(X)
A_std = np.column_stack([np.ones(len(X)), (X - mu) / sd])
w_star = np.linalg.solve(A_raw.T @ A_raw, A_raw.T @ y)
mse_star = np.mean((A_raw @ w_star - y) ** 2)


def steps_to(losses, rel=1e-6):
    hit = np.nonzero(losses - mse_star <= rel * mse_star)[0]
    return int(hit[0]) if len(hit) else None


for name, A, n_iter in [("kg, lít", A_raw, 10000), ("chuẩn hoá", A_std, 100)]:
    eig = np.linalg.eigvalsh(2 * A.T @ A / len(y))
    lr = 1 / eig[-1]
    _, losses = gradient_descent(A, y, lr, n_iter)
    print(f"{name:10s} kappa(H) = {eig[-1] / eig[0]:8.1f}  lr tối đa = {2 / eig[-1]:.5f}  "
          f"số bước tới 1e-6 với lr = 1/L: {steps_to(losses)}")
print("normal equation:", w_star.round(4), "MSE train:", round(mse_star, 3))

In [ ]:
# explore: vượt ngưỡng 2/L thì phân kỳ; schedule inverse_time cứu được lr0 hơi lớn
L = np.linalg.eigvalsh(2 * A_std.T @ A_std / len(y)).max()
runs = {
    "0.9 · 2/L constant": dict(lr0=0.9 * 2 / L),
    "1.1 · 2/L constant": dict(lr0=1.1 * 2 / L),
    "1.1 · 2/L inverse_time": dict(lr0=1.1 * 2 / L, schedule="inverse_time", decay=0.05),
    "0.1 · 2/L constant": dict(lr0=0.1 * 2 / L),
}
for label, kw in runs.items():
    _, losses = gradient_descent(A_std, y, n_iter=100, **kw)
    plt.semilogy(losses - mse_star + 1e-9, label=label)
    print(f"{label:24s} loss sau 100 bước: {losses[-1]:.4g}")
plt.xlabel("bước")
plt.ylabel("loss − loss tối ưu")
plt.ylim(1e-8, 1e6)
plt.legend()
plt.show()

In [ ]:
# explore: hệ số trên đơn vị gốc và chất lượng trên test (đánh giá đúng một lần, cuối cùng)
w_std, _ = gradient_descent(A_std, y, lr0=0.25, n_iter=300)
w = to_original_scale(w_std, mu, sd)
print("GD (đơn vị gốc):", w.round(4))
test = table[table.split == "test"]
pred = np.column_stack([np.ones(len(test)), test[FEATURES].to_numpy()]) @ w
resid = test.freight_value.to_numpy() - pred
baseline = test.freight_value.to_numpy() - y.mean()
print(f"test RMSE {np.sqrt(np.mean(resid ** 2)):.2f}  (dự đoán bằng trung bình train: {np.sqrt(np.mean(baseline ** 2)):.2f})")
print(f"test R² {1 - resid @ resid / np.sum((test.freight_value - test.freight_value.mean()) ** 2):.3f}")

## 6. Nhận xét (bài mẫu)

- **Condition number quyết định tốc độ.** Trên đơn vị kg/lít, Hessian có $$\kappa \approx 1329$$ nên
  với lr = 1/L gradient descent cần 9 231 bước để loss cách tối ưu dưới 1e-6 (tương đối); chuẩn hoá
  đưa $$\kappa$$ về khoảng 8.6 và chỉ cần 30 bước. Cùng một bài toán, cùng nghiệm.
- **Ngưỡng 2/L là thật.** Với lr = 1.1 · 2/L loss nổ tung theo hàm mũ; 0.9 · 2/L vẫn hội tụ (dao động
  dấu ở hướng có độ cong lớn nhất). Schedule `inverse_time` kéo lr xuống dưới ngưỡng sau vài bước:
  loss nhích lên ở bước thứ hai (267 → 275) rồi vẫn hội tụ về 111.9.
- **Nghiệm khớp normal equation**: freight ≈ 13.25 + 1.46 · kg + 0.18 · lít.
  Mỗi kg thêm khoảng 1.46 BRL, mỗi lít thêm khoảng 0.18 BRL, giữ feature kia cố định.
- **Model yếu vì thiếu feature, không phải vì optimizer.** Test RMSE 14.87 so với 18.40 của dự
  đoán hằng; R² test 0.342 thấp hơn train 0.458. Phí ship phụ thuộc mạnh vào quãng đường seller–khách
  (bang), thứ chưa có ở đây. Tối ưu tốt hơn không cứu được model thiếu thông tin.